# Audio Extraction

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
print(f'\tOPENAI_API_KEY={os.getenv("OPENAI_API_KEY")[:20]}')

	OPENAI_API_KEY=sk-proj-DPrT3N-1Ilp9


In [2]:
# step1 : 동영상 -> 오디오 추출
# step2 : 오디오를 10분 단위로 분할
# step3 : 각 분할 오디오로 Whisper API 호출 -> 전사문 (transcript) 
# step4 : 모든 전사문 합치기

## ffmpeg 사용

In [ ]:
# CLI 에서 실행해보자.

# >ffmpeg -i mp4파일명 -vn mp3파일명 -y
# -i 옵션 : input. 입력 파일 지정 (여러개 지정 가능)
# -vn 옵션 : disable video. 비디오는 무시하고 오디오만 추출
# -y 옵션 : 덮어쓰기 할때 y/n 사용자 입력 대디하지 않고 yes 로 진행

# 실행결과=> autio.mp3 생성됨  <- 폴더에서 확인해보자.


In [3]:
import subprocess

In [12]:
base_path = r'/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4'  # 동영상 경로
out_path = r'/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/out' # 분할 오디오 추출 경로

import os
if not os.path.exists(out_path):
    os.makedirs(out_path)

video_file = 'podcast.mp4'
audio_file = 'audio.mp3'

src_path = os.path.join(base_path, video_file)
dst_path = os.path.join(base_path, audio_file)

print(src_path)
print(dst_path)


/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/podcast.mp4
/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/audio.mp3


## [👉🏻 extract_audio_from_video() 함수]

In [14]:
# video_path: 동영상 경로
# audio_path: 추출한 오디오 경로
def extract_audio_from_video(video_path, audio_path):
    # CLI 에서 실행할 command 준비
    # 커맨드창 (CLI) 에서 명령은 아래와 같다.
    #  > ffmpeg -i D:\dataset\podcast.mp4 -vn D:\NLP2501\dataset\audio.mp3 -y

    command = ["ffmpeg", "-i", video_path, "-vn", audio_path, "-y"]
    subprocess.run(command)

In [16]:
extract_audio_from_video(src_path, dst_path)

ffmpeg version 8.1.2 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 21.0.0 (clang-2100.0.123.102)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.1.2 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60. 26.102 / 60. 26.102
  libavcodec     62. 28.102 / 62. 28.102
  libavformat    62. 12.102 / 62. 12.102
  libavdevice    62.  3.102 / 62.  3.102
  libavfilter    11. 14.102 / 11. 14.102
  libswscale      9.  5.102 /  9.  5.102
  libswresample   6.  3.102 /  6.  3.102
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/podcast.mp4':
  Metadata:
    major_brand     : mp42
    minor_version   : 0
   

# Cutting The Audio

## pydub 패키지

In [ ]:
# pydub 패키지
#  공식: https://github.com/jiaaro/pydub 
#  pip install pydub  <- 설치 필요  (사전에 ffmpeg 가 설치되고 경로 설정도 되어 있어야 한다)


In [ ]:
# Whisper 모델의 입력 한도 : 10분

# 10분길이의 mp3 파일들로 분할해야 한다.

In [17]:
from pydub import AudioSegment

In [18]:
track = AudioSegment.from_mp3(dst_path)

In [20]:
# track

In [21]:
track.duration_seconds

4422.426122448979

In [23]:
len(track)  # ms

4422426

In [25]:
five_minutes = 5 * 60 * 1000

first_five = track[:five_minutes]  # 첫 5분 선택

In [27]:
# first_five

In [28]:
first_five.duration_seconds

300.0

In [30]:
first_five.export(os.path.join(out_path, 'first_five.mp3'), format='mp3')

<_io.BufferedRandom name='/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/out/first_five.mp3'>

## [👉🏻 cut_audio_in_chunks() 함수]

In [31]:
import math

# audio_path : 원본 오디오 경로
# chunk_size : minute
# chunks_folder: chunk 들을 저장할 폴더
def cut_audio_in_chunks(audio_path, chunk_size, chunks_folder):
    track = AudioSegment.from_mp3(audio_path)

    chunk_len = chunk_size * 60 * 1000
    chunks = math.ceil(len(track) / chunk_len)  # 분할할 파일 개수

    for i in range(chunks):
        start_time = i * chunk_len
        end_time = (i + 1) * chunk_len

        chunk = track[start_time:end_time]

        exp_path = os.path.join(chunks_folder, f"chunk_{i}.mp3")
        chunk.export(exp_path, format='mp3')
    

In [32]:
cut_audio_in_chunks(dst_path, 10, out_path)

In [37]:
ls -all /Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/out/chunk*.mp3

-rw-r--r--@ 1 leo  staff  9600775 Jun 26 16:09 /Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/out/chunk_0.mp3
-rw-r--r--@ 1 leo  staff  9600775 Jun 26 16:09 /Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/out/chunk_1.mp3
-rw-r--r--@ 1 leo  staff  9600775 Jun 26 16:09 /Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/out/chunk_2.mp3
-rw-r--r--@ 1 leo  staff  9600775 Jun 26 16:09 /Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/out/chunk_3.mp3
-rw-r--r--@ 1 leo  staff  9600775 Jun 26 16:09 /Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/out/chunk_4.mp3
-rw-r--r--@ 1 leo  staff  9600775 Jun 26 16:09 /Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/out/chunk_5.mp3
-rw-r--r--@ 1 leo  staff  9600775 Jun 26 16:09 /Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/out/chunk_6.mp3
-rw-r--r--@ 1 leo  staff  3559593 Jun 26 16:09 /Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/out/chun

# Whisper Transcript

In [39]:
import openai

In [40]:
transcript = openai.audio.transcriptions.create(
    model='whisper-1',
    file=open(os.path.join(out_path,"chunk_0.mp3"), "rb"),
    language="en",
)

transcript

Transcription(text="If success is this lagging indicator of commitment now, how can you be sure that you are paying your dues? The best-selling author and host. The number one health and wellness podcast. On Purpose with Jay Shetty. Society has gone in the direction of becoming addicted to pleasure. Yes. Or pleasure-seeking. Where, from the Stoic's perspective, why did we even ever go down that road? Like, why did we leave wisdom and self-control? Or did we never have it at all and we've always been trying to balance it? Yeah. I mean, I guess that's the big question is like, why do we take something that we like too far? Yeah. Right. So the Epicureans would say like, look, drinking is great, but if you have a hangover the next day, was it actually so great? And so, you know, if you if you push the pleasure too far, it becomes not pleasurable. But in the moment, that feels very far away. Right. Like in the moment you want the thing now. Obviously, sex is this thing for people. It's like

## [👉🏻 transcribe_chunks() 함수]

In [50]:
import glob
def transcribe_chunks(chunk_folder, destination):
    files = glob.glob(os.path.join(chunk_folder, "chunk*.mp3"))
    files.sort()

    for file in files:
        with open(file, "rb") as audio_file, open(destination, "a") as text_file:
            print(file, '녹취록 가져오는 중...', end='')
            transcript = openai.audio.transcriptions.create(
                model='whisper-1',
                file=audio_file,
                language="en",
            )
            text_file.write(transcript.text)
            print('완료')

In [51]:
transcribe_chunks(out_path, os.path.join(out_path, 'transcript.txt'))

/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/out/chunk_0.mp3 녹취록 가져오는 중...완료
/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/mp4/out/chunk_1.mp3 녹취록 가져오는 중...완료
